# Genetic Algorithm Hyperparameter Tuning
This notebook will show the outputs of the hyperparameter tuning. So this will be done in stages rather than a full grid search. In addition this will be done for only the moderate city, this is because if we used all three cities during tuning would conflate city structure with hyperparameter effects. You wouldn't know if a parameter setting is better because it's genuinely better or just better suited to that specific city.

Moderate city is the right choice for tuning because it's the middle ground — grid is too simple and bottleneck is too constrained. Once you've 

In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
import pickle

def extract_results(path):
    with open(path, "rb") as f:
        results = pickle.load(f)
    
    return results

In [19]:
import numpy as np
import pandas as pd


def summarise_experiment(results: dict, experiment_name: str) -> dict:
    evals = results["evals"]
    all_metrics = [e.metrics for e in evals]

    summary = {"experiment": experiment_name}
    for metric in all_metrics[0].keys():
        values = [m[metric] for m in all_metrics]
        summary[f"{metric} (mean)"] = round(float(np.mean(values)), 3)
        summary[f"{metric} (std)"] = round(float(np.std(values)), 3)

    return summary


def compare_experiments(filepaths: dict) -> pd.DataFrame:
    """
    filepaths: dict of {experiment_name: filepath}
    e.g. {
        "fitness_max": "results/fitness_max.pkl",
        "fitness_mean": "results/fitness_mean.pkl",
        "fitness_05max_05mean": "results/fitness_combo_05.pkl",
    }
    """
    rows = []
    for name, path in filepaths.items():
        results = extract_results(path)
        summary = summarise_experiment(results, name)
        rows.append(summary)

    df = pd.DataFrame(rows).set_index("experiment")
    return df


def print_comparison(df: pd.DataFrame, sort_by: str = "Total time (mean)") -> None:
    sorted_df = df.sort_values(sort_by)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)
    print(sorted_df.to_string())

### Fitness Function
The following parameters were tried first:
```
{"fitness": "max", "alpha": 1},
{"fitness": "mean", "alpha": 1},
{"fitness": "median", "alpha": 1},
{"fitness": "max-median", "alpha": 0.8},
{"fitness": "max-median", "alpha": 0.65},
{"fitness": "max-median", "alpha": 0.5}
```

Saved under the following name:
```
path = f"outputs/tuning/{city.city_name}_{n_experiments}_{params['fitness']}_{params['alpha']}_fitness.pkl"

>>outputs/tuning/moderate_city_3_{params['fitness']}_{params['alpha']}_fitness.pkl

```

In [22]:
filepaths = {
    "fitness_max":          "outputs/tuning/moderate_city_3_max_1_fitness.pkl",
    "fitness_mean":         "outputs/tuning/moderate_city_3_mean_1_fitness.pkl",
    "fitness_median":         "outputs/tuning/moderate_city_3_median_1_fitness.pkl",
    "fitness_08max_02median": "outputs/tuning/moderate_city_3_max-median_0.8_fitness.pkl",
    "fitness_065max_035median":       "outputs/tuning/moderate_city_3_max-median_0.65_fitness.pkl",
    "fitness_05max_05median": "outputs/tuning/moderate_city_3_max-median_0.5_fitness.pkl",
}


In [ ]:
df = compare_experiments(filepaths)
print_comparison(df, sort_by="Total time (mean)")

                          Avg path length (mean)  Avg path length (std)  Total time (mean)  Total time (std)  Average time (mean)  Average time (std)  Congestion index (mean)  Congestion index (std)  Avg Congestion delay (mean)  Avg Congestion delay (std)  Exit utilisation (mean)  Exit utilisation (std)  Avg Path Efficiency (mean)  Avg Path Efficiency (std)
experiment                                                                                                                                                                                                                                                                                                                                                             
fitness_08max_02median                     8.980                  0.073             16.667             0.471               11.367               0.209                    0.857                   0.031                        2.387                       0.144                    0.092

In [31]:
df[['Avg path length (mean)', 'Total time (mean)', 'Average time (mean)', 'Congestion index (mean)', 
    'Avg Congestion delay (mean)', 'Exit utilisation (mean)', 'Avg Path Efficiency (mean)']].sort_values('Avg Congestion delay (mean)')


,Avg path length (mean),Total time (mean),Average time (mean),Congestion index (mean),Avg Congestion delay (mean),Exit utilisation (mean),Avg Path Efficiency (mean)
experiment,,,,,,,
fitness_mean,8.187,17.667,10.493,0.833,2.307,0.100,1.105
fitness_median,8.970,25.333,11.277,0.830,2.307,0.112,1.198
fitness_05max_05median,8.997,17.000,11.360,0.840,2.363,0.075,1.180
fitness_08max_02median,8.980,16.667,11.367,0.857,2.387,0.092,1.164
fitness_065max_035median,8.833,17.000,11.260,0.827,2.427,0.105,1.150
fitness_max,9.207,17.000,11.640,0.827,2.433,0.065,1.182


Clear winner is fitness_08max_02mean:
* Lowest total time (16.667) — the only config that gets below 17
* Lowest std on total time (0.471) — most consistent across seeds
* Lowest avg congestion delay (2.387)
* Best path efficiency (1.164) — agents taking reasonably direct routes

A few other observations:
* fitness_mean has the best average time (10.493) and shortest paths (8.187) but total time is worst (17.667) — it's optimising average at the expense of stragglers, which is wrong for evacuation
* fitness_median is clearly worst — total time 25.333 is terrible, drop it entirely
* fitness_065max has the best exit utilisation (0.105) but total time is inconsistent (std 0.816)
* fitness_max is completely flat (std 0.000) — confirms the earlier finding that pure max gives no gradient

### Epsilon
The following parameters were tried for the mutation greedy path and utilising the above parameters for fitness:
```
epsilon - [0, 0.2, 0.4, 0.6, 0.8, 1]
```

Saved under the following name:
```
path = f"outputs/tuning/{city.city_name}_{n_experiments}_{params['epsilon']}_epsilon.pkl"
>>outputs/tuning/moderate_city_3_{params['epsilon']}_epsilon.pkl
```

In [ ]:
epsilon_filepaths = {
    "epsilon_0": "outputs/tuning/moderate_city_3_0_epsilon.pkl",
    "epsilon_02": "outputs/tuning/moderate_city_3_02_epsilon.pkl",
    "epsilon_04": "outputs/tuning/moderate_city_3_04_epsilon.pkl",
    "epsilon_06": "outputs/tuning/moderate_city_3_06_epsilon.pkl",
    "epsilon_08": "outputs/tuning/moderate_city_3_08_epsilon.pkl",
    "epsilon_1": "outputs/tuning/moderate_city_3_1_epsilon.pkl",
}

df_epsilon = compare_experiments(epsilon_filepaths)
df_epsilon[['Avg path length (mean)', 'Total time (mean)', 'Average time (mean)', 'Congestion index (mean)', 
    'Avg Congestion delay (mean)', 'Exit utilisation (mean)', 'Avg Path Efficiency (mean)']
           ].sort_values('Total time (mean)')

            Avg path length (mean)  Avg path length (std)  Total time (mean)  Total time (std)  Average time (mean)  Average time (std)  Congestion index (mean)  Congestion index (std)  Avg Congestion delay (mean)  Avg Congestion delay (std)  Exit utilisation (mean)  Exit utilisation (std)  Avg Path Efficiency (mean)  Avg Path Efficiency (std)
experiment                                                                                                                                                                                                                                                                                                                                               
epsilon_1                    8.703                  0.092             16.333             0.471               11.100               0.225                    0.860                   0.014                        2.397                       0.135                    0.107                   0.039                  

`epsilon = 1.0`

Greedy makes two decisions at every step:
1. Which exit to head to — always the nearest one
2. Which node to move to next — always the one closest to that exit

With epsilon=1.0 the GA mutation only affects decision 2 — path efficiency once an exit is chosen. Decision 1 — which exit — is entirely controlled by the GA's selection, crossover, and mutation exit assignment. The GA is still freely routing agents to exits that aren't the nearest one, which is the entire point.

### CrossOver 
The following parameters were tried for the mutation greedy path and utilising the above parameters for fitness:
```
crossover - [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
```

Saved under the following name:
```
path = f"outputs/tuning/{city.city_name}_{n_experiments}_{params['crossover']}_crossover.pkl"
>>outputs/tuning/moderate_city_3_{params['crossover']}_crossover.pkl
```

In [43]:
crossover_filepaths = {
    "crossover_04": "outputs/tuning/moderate_city_3_04_crossover.pkl",
    "crossover_05": "outputs/tuning/moderate_city_3_05_crossover.pkl",
    "crossover_06": "outputs/tuning/moderate_city_3_06_crossover.pkl",
    "crossover_07": "outputs/tuning/moderate_city_3_07_crossover.pkl",
    "crossover_08": "outputs/tuning/moderate_city_3_08_crossover.pkl",
    "crossover_09": "outputs/tuning/moderate_city_3_09_crossover.pkl",
}

df_crossover = compare_experiments(crossover_filepaths)
df_crossover[['Avg path length (mean)', 'Total time (mean)', 'Average time (mean)', 'Congestion index (mean)', 
    'Avg Congestion delay (mean)', 'Exit utilisation (mean)', 'Avg Path Efficiency (mean)']
           ].sort_values('Total time (mean)')


,Avg path length (mean),Total time (mean),Average time (mean),Congestion index (mean),Avg Congestion delay (mean),Exit utilisation (mean),Avg Path Efficiency (mean)
experiment,,,,,,,
crossover_08,8.713,15.667,11.207,0.877,2.493,0.145,1.135
crossover_09,8.653,16.000,11.040,0.840,2.387,0.097,1.114
crossover_04,8.950,16.333,11.457,0.860,2.507,0.058,1.167
crossover_05,8.680,16.333,11.103,0.843,2.423,0.100,1.127
crossover_07,8.523,16.333,10.920,0.843,2.397,0.088,1.128
crossover_06,8.813,16.333,11.167,0.827,2.353,0.097,1.148


Crossover probability 0.8 shows the best results across the total time.

`Crossover 0.8`

### Mutation 
The following parameters were tried for the mutation greedy path and utilising the above parameters for fitness:
```
mutation - [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
```

Saved under the following name:
```
path = f"outputs/tuning/{city.city_name}_{n_experiments}_{params['mutation']}_mutation.pkl"
>>outputs/tuning/moderate_city_3_{params['mutation']}_mutation.pkl
```

In [46]:
mutation_filepaths = {
    "mutation_001": "outputs/tuning/moderate_city_3_001_mutation.pkl",
    "mutation_005": "outputs/tuning/moderate_city_3_005_mutation.pkl",
    "mutation_01": "outputs/tuning/moderate_city_3_01_mutation.pkl",
    "mutation_02": "outputs/tuning/moderate_city_3_02_mutation.pkl",
    "mutation_03": "outputs/tuning/moderate_city_3_03_mutation.pkl",
    "mutation_04": "outputs/tuning/moderate_city_3_04_mutation.pkl",
}

df_mutation = compare_experiments(mutation_filepaths)
df_mutation[['Avg path length (mean)', 'Total time (mean)', 'Average time (mean)', 'Congestion index (mean)', 
    'Avg Congestion delay (mean)', 'Exit utilisation (mean)', 'Avg Path Efficiency (mean)']
           ].sort_values('Total time (mean)')


,Avg path length (mean),Total time (mean),Average time (mean),Congestion index (mean),Avg Congestion delay (mean),Exit utilisation (mean),Avg Path Efficiency (mean)
experiment,,,,,,,
mutation_001,8.357,15.000,10.830,0.877,2.473,0.088,1.116
mutation_005,8.570,16.000,10.880,0.833,2.310,0.093,1.133
mutation_01,8.623,16.000,11.100,0.837,2.477,0.147,1.148
mutation_02,8.867,16.333,11.317,0.853,2.450,0.087,1.129
mutation_03,8.863,16.667,11.350,0.850,2.487,0.098,1.133
mutation_04,8.790,17.000,11.210,0.870,2.420,0.073,1.147


Need to double check these have converged so running the same again with `_100` on the end to represent the increase in max evolutions.

In [50]:
mutation_filepaths_100 = {
    "mutation_001": "outputs/tuning/moderate_city_3_001_mutation_100.pkl",
    "mutation_005": "outputs/tuning/moderate_city_3_005_mutation_100.pkl",
    "mutation_01": "outputs/tuning/moderate_city_3_01_mutation_100.pkl",
    "mutation_02": "outputs/tuning/moderate_city_3_02_mutation_100.pkl",
    "mutation_03": "outputs/tuning/moderate_city_3_03_mutation_100.pkl",
    # "mutation_04": "outputs/tuning/moderate_city_3_04_mutation_100.pkl",
}

df_mutation_100 = compare_experiments(mutation_filepaths_100)
df_mutation_100[['Avg path length (mean)', 'Total time (mean)', 'Average time (mean)', 'Congestion index (mean)', 
    'Avg Congestion delay (mean)', 'Exit utilisation (mean)', 'Avg Path Efficiency (mean)']
           ].sort_values('Total time (mean)')


,Avg path length (mean),Total time (mean),Average time (mean),Congestion index (mean),Avg Congestion delay (mean),Exit utilisation (mean),Avg Path Efficiency (mean)
experiment,,,,,,,
mutation_001,8.537,15.000,10.937,0.840,2.400,0.125,1.126
mutation_005,8.540,15.667,10.927,0.840,2.387,0.110,1.116
mutation_01,8.820,16.333,11.213,0.837,2.393,0.082,1.156
mutation_02,8.747,16.333,11.180,0.853,2.433,0.088,1.129
mutation_03,8.780,16.333,11.217,0.850,2.437,0.070,1.145


**Evolutions**

Tuning runs: 50 evolutions — sufficient to differentiate configs, validated by comparing 50 vs 100 evolution results which showed consistent rankings
Final experiments: 100 evolutions — gives best solutions time to converge

**Mutation rate**

Selected: `mutation=0.05`
Rejected mutation=0.01 despite better total time (15.000 vs 15.667) because with 100 agents only 63% of chromosomes mutate per generation, limiting exploration. mutation=0.05 ensures 99.4% of chromosomes mutate each generation which is more theoretically sound and consistent with standard GA literature
Rejected mutation=0.1 and above as performance degrades monotonically beyond 0.05

### Population Size 
The following parameters were tried for the population size:
```
pop_size - [5, 10, 20, 50, 100, 200]
```

Saved under the following name:
```
path = f"outputs/tuning/{city.city_name}_{n_experiments}_{pop_size}_pop_size.pkl"
```

In [52]:
pop_size_filepaths = {
    "pop_size_5": "outputs/tuning/moderate_city_3_5_pop_size.pkl",
    "pop_size_10": "outputs/tuning/moderate_city_3_10_pop_size.pkl",
    "pop_size_20": "outputs/tuning/moderate_city_3_20_pop_size.pkl",
    "pop_size_50": "outputs/tuning/moderate_city_3_50_pop_size.pkl",
    "pop_size_100": "outputs/tuning/moderate_city_3_100_pop_size.pkl",
    "pop_size_200": "outputs/tuning/moderate_city_3_200_pop_size.pkl",
}

df_pop_size = compare_experiments(pop_size_filepaths)
# df_pop_size[['Avg path length (mean)', 'Total time (mean)', 'Average time (mean)', 'Congestion index (mean)', 
#     'Avg Congestion delay (mean)', 'Exit utilisation (mean)', 'Avg Path Efficiency (mean)']
#            ].sort_values('Total time (mean)')
df_pop_size

,Avg path length (mean),Avg path length (std),Total time (mean),Total time (std),Average time (mean),Average time (std),Congestion index (mean),Congestion index (std),Avg Congestion delay (mean),Avg Congestion delay (std),Exit utilisation (mean),Exit utilisation (std),Avg Path Efficiency (mean),Avg Path Efficiency (std)
experiment,,,,,,,,,,,,,,
pop_size_5,9.297,0.246,20.667,0.471,11.803,0.313,0.853,0.009,2.507,0.132,0.100,0.036,1.233,0.031
pop_size_10,8.973,0.137,17.000,0.000,11.433,0.159,0.847,0.009,2.460,0.171,0.092,0.012,1.172,0.005
pop_size_20,8.553,0.209,16.000,0.000,10.973,0.162,0.843,0.017,2.420,0.107,0.148,0.013,1.131,0.016
pop_size_50,8.667,0.239,15.667,0.471,10.993,0.351,0.843,0.017,2.327,0.158,0.080,0.019,1.132,0.020
pop_size_100,8.407,0.161,15.333,0.471,10.717,0.254,0.840,0.008,2.310,0.098,0.057,0.002,1.100,0.022
pop_size_200,8.183,0.066,15.000,0.000,10.513,0.017,0.850,0.008,2.330,0.050,0.093,0.027,1.082,0.011


`Population size of 50` was selected as it achieves competitive total evacuation time (15.667) with manageable computational cost, with larger populations showing diminishing returns — `pop_200` improves total time by only 0.667 timesteps at four times the computational expense.